# Qwen2.5-0.5B with pairwise data samples

In [1]:
!pip install -q transformers peft evaluate tomli scikit-learn pandas tqdm torch accelerate bitsandbytes
!pip install -U bitsandbytes

In [2]:
import os
from tqdm import tqdm
import random
import json
import re

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.optim import AdamW

from transformers import AutoModel, BitsAndBytesConfig, get_cosine_schedule_with_warmup, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

2026-05-04 16:41:52.567677: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 16:41:52.632595: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/pytho

In [9]:
# config

# TODO: change lora targets of when changing llm type (llama, qwen, ...)


#TRAIN_PATH = "../output/preprocessed_data/english_train_with_evidence.jsonl"
# for pairwise use original data
TRAIN_PATH = "../data/english/english_train.json"
TEST_PATH = "../data/english/clef2026_gpt4_o_mini_val.json"

# Separate teacher output dirs
EXPERIMENT_NAME = "qwen05_pairwise_fixed2"
PRED_FILE_NAME = "clef_predictions.json"

EXPERIMENT_DIR = f"../output/{EXPERIMENT_NAME}/"
RESULT_DIR = f"../output/{EXPERIMENT_NAME}/results/"
PRED_PATH = f"../output/{EXPERIMENT_NAME}/results/{PRED_FILE_NAME}"


TEACHER_MODEL = "meta-llama/Llama-3.2-1B"
BASE_MODEL = "Qwen/Qwen2.5-0.5B" # "meta-llama/Llama-3.2-1B"

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
RANDOM_STATE = 42

#os.makedirs("output/training_data_for_RM", exist_ok=True)
#os.makedirs("output/RM_prediction", exist_ok=True)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

print("Device:", DEVICE)
print("Train file exists:", os.path.exists(TRAIN_PATH))
print("Val file exists:", os.path.exists(TEST_PATH))


Device: cuda
Train file exists: True
Val file exists: True


In [ ]:
# helper functions
def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def get_evidence(sample):
    possible_keys = [
        "evidences",
        "evidence",
        "Evidence",
        "relevant_evidence",
        "Relevant_evidence",
        "context",
        "Context",
        "gold_evidence",
        "Gold_evidence",
    ]

    for key in possible_keys:
        if key in sample and sample[key]:
            value = sample[key]

            if isinstance(value, list):
                return " ".join(map(str, value))

            if isinstance(value, dict):
                return json.dumps(value, ensure_ascii=False)

            return str(value)

    return ""

def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_params} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )

def build_teacher_input(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}\n"
        f"Evidence: {evidence}"
    )

In [11]:
class PairwiseDataset(Dataset):
    def __init__(
        self,
        df,
        tokenizer,
        max_length,
        max_pairs_per_claim=8,
    ):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        dropped = 0
        kept = 0

        for _, row in df.iterrows():
            claim = row["claim"]
            evidence = " ".join(row["evidences"])

            traces = row["Reasoning_traces"]
            verdicts = [l.lower() for l in row["Verdict_list"]]
            gt_label = row["label"].lower() 
            
            # split traces
            correct = [t for t, v in zip(traces, verdicts) if v == gt_label]
            incorrect = [t for t, v in zip(traces, verdicts) if v != gt_label]

            # need both sides
            if len(correct) == 0 or len(incorrect) == 0:
                dropped += 1
                continue

            kept += 1

            pairs = []

            # build correct - incorrect pairs
            for c in correct:
                for i, v in zip(traces, verdicts):
                    if v != gt_label:
                        pairs.append((c, i, gt_label, v))

            # reduce explosion
            if len(pairs) > max_pairs_per_claim:
                pairs = random.sample(pairs, max_pairs_per_claim)

            for pos_trace, neg_trace, pos_label, neg_label in pairs:
                pos_text = build_teacher_input(
                    claim=claim,
                    evidence=evidence,
                    verdict=pos_label,
                    justification=pos_trace,
                )

                neg_text = build_teacher_input(
                    claim=claim,
                    evidence=evidence,
                    verdict=neg_label,
                    justification=neg_trace,
                )

                self.samples.append((pos_text, neg_text))

        print(f"Kept claims: {kept}")
        print(f"Dropped claims: {dropped}")
        print(f"Total training pairs: {len(self.samples)}")
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pos_text, neg_text = self.samples[idx]

        pos_enc = self.tokenizer(
            pos_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        neg_enc = self.tokenizer(
            neg_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "pos_input_ids": pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids": neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
        }

In [ ]:
class CustomModel(torch.nn.Module):
    def __init__(
            self,
            model_name,
            is_teacher=False,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
            use_quant=False
    ):
        super().__init__()#

        if use_quant:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16
            )
            self.model = AutoModel.from_pretrained(model_name, quantization_config=bnb_config,)
        else:
            self.model = AutoModel.from_pretrained(model_name)

        if is_teacher:
            self.model.eval()  # freeze teacher
            for p in self.model.parameters():
                p.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                lora_dropout=0.05,
                bias="none",
            )

            self.model = get_peft_model(self.model, lora_config)

            if use_quant:
                # set kbit training if using quantization AND lora
                self.model = prepare_model_for_kbit_training(self.model)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)


    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        
        logits = self.classifier(pooled_output.float())

        return logits

In [ ]:
class TrainerModule:
    def __init__(
        self,
        model,
        teacher,
        train_loader,
        val_loader,
        epochs,
        lr,
        output_dir,
        use_distillation=False,
        alpha=0.5,
        temperature=2.0,
        margin=0.5,
    ):
        self.device = DEVICE
        self.model = model.to(self.device)

        self.teacher = teacher.to(self.device) if teacher else None
        self.use_distillation = use_distillation and teacher is not None

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.alpha = alpha
        self.temperature = temperature
        self.margin = margin

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self.best_val_acc = 0
    
    def ranking_loss(self, pos_scores, neg_scores, margin):
        diff = pos_scores - neg_scores
        return -torch.mean(torch.log(torch.sigmoid(diff - margin)))

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                pos_ids = batch["pos_input_ids"].to(self.device)
                pos_mask = batch["pos_attention_mask"].to(self.device)

                neg_ids = batch["neg_input_ids"].to(self.device)
                neg_mask = batch["neg_attention_mask"].to(self.device)

                # forward
                pos_scores = self.model(pos_ids, pos_mask).squeeze(-1)
                neg_scores = self.model(neg_ids, neg_mask).squeeze(-1)

                # distillation
                student_diff = pos_scores - neg_scores

                rank_loss = self.ranking_loss(pos_scores, neg_scores, self.margin)
                
                #loss = torch.mean(F.relu(1.0 - (pos_score - neg_score)))

                # distillation 
                if self.use_distillation:
                    with torch.no_grad():
                        t_pos = self.teacher(pos_ids, pos_mask).squeeze(-1)
                        t_neg = self.teacher(neg_ids, neg_mask).squeeze(-1)

                    teacher_diff = t_pos - t_neg

                    student_prob = torch.sigmoid(student_diff / self.temperature)
                    teacher_prob = torch.sigmoid(teacher_diff / self.temperature)

                    kd_loss = F.mse_loss(student_prob, teacher_prob)
                
                    loss = self.alpha * rank_loss + (1 - self.alpha) * kd_loss
                else:
                    loss = rank_loss
                    

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                acc = (pos_scores > neg_scores).float().mean().item()
                total_acc += acc

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Pairwise Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                pos_ids = batch["pos_input_ids"].to(self.device)
                pos_mask = batch["pos_attention_mask"].to(self.device)

                neg_ids = batch["neg_input_ids"].to(self.device)
                neg_mask = batch["neg_attention_mask"].to(self.device)

                pos_scores = self.model(pos_ids, pos_mask).squeeze(-1)
                neg_scores = self.model(neg_ids, neg_mask).squeeze(-1)

                loss = self.ranking_loss(pos_scores, neg_scores,  self.margin)

                total_loss += loss.item()
                total_acc += (pos_scores > neg_scores).float().mean().item()

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Pairwise Acc: {total_acc / len(self.val_loader):.4f}")

        # save only best model
        if total_acc > self.best_val_acc:
            self.best_val_acc = total_acc
            
            torch.save(
                self.model.state_dict(),
                os.path.join(self.output_dir, "best_model.pt"),
            )


In [ ]:

train_df = pd.read_json(TRAIN_PATH,)  #lines=True

train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = PairwiseDataset(train_split, tokenizer, MAX_LENGTH)
val_dataset = PairwiseDataset(val_split, tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
)

student_model = CustomModel(
    model_name=BASE_MODEL,
    use_lora=True,
    lora_rank=8,
    lora_alpha=16,
)

print_trainable_parameters(student_model)

# teacher_model = CustomModel(
#     model_name=TEACHER_MODEL,
#     is_teacher=True,  
#     use_lora=False,
#     use_quant=True
# )

student_model.model.config.pad_token_id = tokenizer.pad_token_id
#teacher_model.model.config.pad_token_id = tokenizer.pad_token_id

trainer = TrainerModule(
    model=student_model,
    teacher=None,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    output_dir=EXPERIMENT_DIR,
    use_distillation=False,   # optional
)

trainer.train()

Kept claims: 2376
Dropped claims: 2744
Total training pairs: 19008
Kept claims: 582
Dropped claims: 698
Total training pairs: 4656

Epoch 1/5


100%|██████████| 2376/2376 [26:19<00:00,  1.50it/s]


Train Loss: 0.5577
Train Pairwise Acc: 0.6798
Val Loss: 0.6363
Val Pairwise Acc: 0.6044

Epoch 2/5


100%|██████████| 2376/2376 [26:19<00:00,  1.50it/s]


Train Loss: 0.2763
Train Pairwise Acc: 0.8532
Val Loss: 1.2459
Val Pairwise Acc: 0.5835

Epoch 3/5


...

Train Loss: 0.1340
Train Pairwise Acc: 0.9159
Val Loss: 1.9120
Val Pairwise Acc: 0.5831

Epoch 4/5


100%|██████████| 2376/2376 [26:13<00:00,  1.51it/s]


Train Loss: 0.1140
Train Pairwise Acc: 0.9225
Val Loss: 2.3436
Val Pairwise Acc: 0.5782

Epoch 5/5


100%|██████████| 2376/2376 [26:11<00:00,  1.51it/s]


Train Loss: 0.1113
Train Pairwise Acc: 0.9256
Val Loss: 2.4381
Val Pairwise Acc: 0.5805


In [ ]:
class TeacherEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
    ):
        self.device = DEVICE

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomModel(
            model_name=base_model,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
        )

        state = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state)

        self.model.to(self.device)
        self.model.eval()

    def encode_input(
            self,
            claim,
            evidence,
            verdict,
            justification,
            max_length=MAX_LENGTH,
    ):
        text = build_teacher_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidence, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            output = self.model(input_ids, attention_mask)

            score = output.squeeze()

            if score.ndim > 0:
                score = score.mean()

            return float(score.cpu().item())

In [ ]:
MODEL_PATH = os.path.join(
    EXPERIMENT_DIR,
    "best_model.pt"
)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

teacher_evaluator = TeacherEvaluator(
    model_path=MODEL_PATH,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

predictions = []

for idx, sample in enumerate(tqdm(val_data)):
    claim = sample["claim"]
    evidence = get_evidence(sample)

    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx in range(len(sample["Reasoning_traces"])):
        justification = remove_label_pattern(
            sample["Reasoning_traces"][trace_idx]
        ).split("Label:")[0]

        verdict = sample["Verdict_list"][trace_idx].lower()

        score = teacher_evaluator.score(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        verdict_list.append(sample["Verdict_list"][trace_idx])
        justification_list.append(justification)
        verifier_score_list.append(score)

    # best_idx = int(np.argmax(np.array(verifier_score_list)))
    # best_verdict = verdict_list[best_idx]

    scores = np.asarray(verifier_score_list, dtype=np.float32)
    sorted_indices = np.argsort(-scores)

    TOP_K = 3

    top_indices = sorted_indices[:TOP_K]

    verdict_scores = {}

    for i in top_indices:
        verdict = verdict_list[i].lower()
        score = scores[i]

        if verdict not in verdict_scores:
            verdict_scores[verdict] = 0.0

        #verdict_scores[verdict] += score  # weighted vote
        verdict_scores[verdict] = verdict_scores.get(verdict, 0.0) + float(score)

    final_verdict = max(verdict_scores.items(), key=lambda x: x[1])[0]

    predictions.append(
        {
            "query_id": sample.get("query_id", idx),
            "Claim": claim,
            "Evidence": evidence,
            "Label": sample["label"],
            "Final_Verdict": final_verdict,
            "TopK_indices": top_indices.tolist(),
            "TopK_scores": scores[top_indices].tolist(),
            "Verdict_BoN": final_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list": verifier_score_list,
        }
    )

with open(PRED_PATH, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved teacher predictions to {PRED_PATH}")
print("Number of predictions:", len(predictions))


100%|██████████| 1600/1600 [16:08<00:00,  1.65it/s]


Saved teacher predictions to ../output/qwen05_pairwise_fixed2/results/clef_predictions.json
Number of predictions: 1600
